# EngSVG - LoRA SFT on engineering drawings, scored with FEM**VS Code + Colab extension:** Select Kernel > Colab > Auto Connect, then Run All.The kernel is a Colab GPU VM; outputs are saved into this local file.**Colab web:** Runtime > Change runtime type > GPU, then Run all.Results are printed inline (so they persist in this file) and written to`/content/engsvg-run/` on the VM.

In [ ]:
import subprocess, sysprint(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],                     capture_output=True, text=True).stdout.strip() or 'NO GPU - select a GPU runtime')

In [ ]:
!rm -rf /content/SVG!git clone -q --depth 1 -b cvpr2027-research-package --filter=blob:none --sparse \    https://github.com/ayushdebnath012/SVG.git /content/SVG!cd /content/SVG && git sparse-checkout set cvpr2027/scripts cvpr2027/src \    cvpr2027/data/eng-svg-bench-text cvpr2027/data/eng-svg-bench-edit!pip -q install svgpathtools==1.7.2 peft transformers accelerate scikit-fem# Colab preinstalls torchao 0.10; peft's LoRA dispatch raises on anything below 0.16.!pip uninstall -y -q torchao!cd /content/SVG && git log --oneline -1

In [ ]:
# Regenerate the 800 training pairs deterministically (seed 20260923), then verify them.!cd /content/SVG/cvpr2027 && PYTHONPATH=src:scripts python scripts/eng_svg_trainset.py build --n 400!cd /content/SVG/cvpr2027 && PYTHONPATH=src:scripts python scripts/eng_svg_trainset.py verify

In [ ]:
# Base vs trained on 24 held-out text2svg tasks, scored by the real benchmark scorer.!cd /content/SVG/cvpr2027 && PYTHONPATH=src:scripts ENGSVG_ROOT=/content/SVG/cvpr2027 \    python scripts/colab_train_engsvg.py --arms text2svg --epochs 2 --eval-count 24 \    --out /content/engsvg-run

In [ ]:
# Print the full summary inline so it is captured in this notebook file.import json, pathlibs = json.loads(pathlib.Path('/content/engsvg-run/summary.json').read_text())for tag in ('before','after'):    r = s[tag]; n = max(r['n'], 1)    print(f"{r['tag']:8s} parsed {r['parsed']:2d}/{r['n']}  geometry {r['geometry_ok']:2d}/{n}  "          f"dimensions {r['dimensions_ok']:2d}/{n}  analysis {r['analysis_ok']:2d}/{n}  "          f"drawnFEM {r['drawn_fem_ok']:2d}/{n}  STRICT {r['strict_pass']:2d}/{n}")print()print(json.dumps({k: v for k, v in s.items() if k not in ('before','after')}, indent=2))